---
# Instructions pour quarto
title: ""
format:
  html:
    code-fold: true
    embed-resources: true
  
---

<!-- Header stylé -->
<div style="background: linear-gradient(90deg, #4e54c8, #8f94fb); padding: 30px; border-radius: 12px; color: white; text-align: center; margin-bottom: 20px;">
  <h3 style="margin: 0; font-size: 3em;">🏠🚆Lien entre l'offre de transport et le prix du logement en Île-de-France 🏠🚆</h3>
</div>

# Introduction

L'objectif de ce projet est d'analyser comment l'accessibilité aux transports en commun influence le prix du logement en Île‑de‑France. Pour cela nous croisons deux sources complémentaires : 
- les transactions immobilières obtenues via la base de donnée DVF ("Demande de valeur foncière") pour obtenir les prix et la localisation des logements
- les données issues de l'API IDFM (Île-de-France Mobilité) décrivant les horaires prévus des transports en commun dans les 30 prochains jours.

Les traitements effectués sont les suivants :

1.Une première phase d'ingestion, nettoyage et visualisation est effectuée séparément chaque base de donnée. Ces phases son présentées dans des pages séparées : [Traitement de IDFM](idfm.html) et [Traitement de DVF](dvf.html)

2. La version nettoyée de DVF est ensuite augmentée d'informations sur la desserte en transport du logement ([Première partie](#ajout-de-la-desserte-à-DVF))

3. Des visualisations sont produites à partir de cette nouvelle base de donnée

4. [PARTIE ECONO] ???

5. Partie ML 

# Ajout de la desserte à DVF

L'approche retenue et d'ajouter au dataframe DVF des métriques indiquant la desserte en transport de chaque logement vendu.


## Calcul des arrêts les plus proches

Après avoir ouvert les fichiers intermédiares, on ajoute à chaque logement la distance à l'arrêt le plus proche pour chaque mode de transport (train, metro, tramway, bus).

Les première lignes du dataframe obtenues sont affichées pour illustrer (seules les colonnes nouvellement ajoutées sont affichées).

In [15]:
import os
import geopandas as gpd

# verifie que l'exectution se fait depuis le bon répertoire
if os.getcwd().endswith("notebooks"):
    os.chdir('..')

# ouverture des fichiers geojson
gdf_idfm_small = gpd.read_file("cache/results/passage_par_arret_synthetique.geojson")
gdf_idfm = gpd.read_file("cache/results/passage_par_arret_full.geojson")
gdf_dvf = gpd.read_file("cache/results/prix_logements.geojson")

# ajout d'une cle principale à gdf_dvf
gdf_dvf['point_id'] = gdf_dvf.index.astype(str)


# reprojeter en CRS métrique, trouver le point le plus proche et la distance en mètres
gdf_dvf_m = gdf_dvf.to_crs(epsg=3857)
gdf_idfm_small_m = gdf_idfm_small.to_crs(epsg=3857)
gdf_idfm_m = gdf_idfm.to_crs(epsg=3857)

for target in ["bus", "metro", "tramway", "train"]:
    target_df = gdf_idfm_small_m[gdf_idfm_small_m[f"nb_{target}_per_day"] > 0]

    nearest = gpd.sjoin_nearest(
        gdf_dvf_m,
        target_df[['stop_id', 'geometry']],
        how='left',
        distance_col='dist_m'
    )

    # ajouter résultats (distance en m et km, id du stop le plus proche) au GeoDataFrame original
    nearest = nearest.reset_index(drop=True)

    gdf_dvf[f"nearest_{target}_stop_id"] = nearest['stop_id']
    gdf_dvf[f"nearest_{target}_dist_m"] = nearest['dist_m']

# afficher le résultat
gdf_dvf[['adresse', 'nearest_bus_stop_id', 'nearest_bus_dist_m', 'nearest_metro_stop_id', 'nearest_metro_dist_m', 'nearest_tramway_stop_id', 'nearest_tramway_dist_m', 'nearest_train_stop_id', 'nearest_train_dist_m']].head()

,adresse,nearest_bus_stop_id,nearest_bus_dist_m,nearest_metro_stop_id,nearest_metro_dist_m,nearest_tramway_stop_id,nearest_tramway_dist_m,nearest_train_stop_id,nearest_train_dist_m
0,1 ALL ADRIENNE,IDFM:73300,155.389997,IDFM:426280,3222.044283,IDFM:73312,708.300133,IDFM:73297,810.600601
1,1 ALL ANDRE MALRAUX,IDFM:74261,264.083932,IDFM:69884,71707.338011,IDFM:68293,52890.886380,IDFM:62168,1451.228854
2,1 ALL ANDRE MALRAUX,IDFM:65153,251.057114,IDFM:71517,22369.191978,IDFM:480927,6986.766072,IDFM:73604,3325.780821
3,1 ALL ANTOINE GROSSIN,IDFM:70505,261.182603,IDFM:70671,2109.592489,IDFM:70310,2226.866100,IDFM:70505,261.182603
4,1 ALL ARAGON,IDFM:73498,154.031537,IDFM:426280,15289.094394,IDFM:73411,7031.585384,IDFM:73482,1142.816999


## Calcul de l'offre de transport dans un rayon de 1km

- **Objectif :** pour chaque point de DVF, compter et agréger l'offre de transport présente dans un rayon de 1 km.
- **Étapes :** 
    - on crée un disque de rayon 1 km autour de chaque logement.
    - on effectue une jointure spatiale pour obtenir tous les couples arrêt situés à ≤ 1 km.
    - dans chauque disque aggrège par ligne de transport (*"route"*) puis par mode (*"route_type"*) pour obtenir le nombre de passage de transport en commun par jour et par mode autour de chaque logement. L'aggrègation d'abord par ligne permet d'éviter de compter plusieurs fois un moyen de transport qui s'arrête plusieurs fois autour d'un logment.
    - après quelques étapes supplémentaires (pivot, renommage, filtrage), les résultats sont ajoutés à DVF

In [16]:
radius_m = 1000 # Distance seuil

# Créer des buffers (disques) de rayon 1 km autour de chaque point DVF
# Utilise gdf_dvf_m (déjà en CRS métrique) pour créer les buffers
gdf_dvf_buffers = gdf_dvf_m.copy()
gdf_dvf_buffers['geometry'] = gdf_dvf_m.geometry.buffer(radius_m)

# Jointure spatiale : trouver tous les couples (point DVF, arrêt de transport) où l'arrêt intersecte le buffer du point
joined = gpd.sjoin(
    gdf_dvf_buffers[['point_id', 'geometry']], 
    gdf_idfm_m[['stop_id', 'route_id', 'nb_stops_per_day', 'route_type', 'geometry']], 
    how='left',  # Jointure gauche pour garder tous les points DVF, même sans intersections
    predicate='intersects' 
).drop(columns=['index_right', 'geometry']).reset_index(drop=True) # Supprimer les colonnes inutiles après jointure


# Agrégation première : par point_id, route_id et route_type, prendre le max de nb_stops_per_day par route
# Cela évite de compter plusieurs fois une route qui passe par plusieurs arrêts dans le rayon
grouped = joined.groupby(
    ['point_id', 'route_id', 'route_type'],
    dropna=False,  # Garder les groupes avec NaN
    as_index=False
).agg(
    nb_stops_per_day_route=('nb_stops_per_day', 'max')  # Max passages par route
).reset_index(drop=True)

# Agrégation seconde : par point_id et route_type, sommer les passages, compter les routes et stations uniques
grouped2 = joined.groupby(['point_id', 'route_type'],
    dropna=False,
    as_index=False).agg(
    passage_journalier=('nb_stops_per_day', 'sum'),  # Somme des passages journaliers par mode
    nb_routes=('route_id', 'nunique'),  # Nombre de routes uniques par mode
    nb_stations=('stop_id', 'nunique')  # Nombre de stations uniques par mode
).reset_index(drop=True)

# Pivot de grouped2 pour avoir une colonne par mode de transport (route_type)
# Les valeurs sont passage_journalier, nb_routes, nb_stations pour chaque mode
grouped2_pivot = grouped2.pivot_table(
    index='point_id',
    columns='route_type', 
    values=['passage_journalier', 'nb_routes', 'nb_stations'],  
    dropna=False,  # Garder les NaN
    fill_value=0  # Remplir les valeurs manquantes par 0 (aucun passage/routes/stations)
)

# Renommer les colonnes pour mapper les codes route_type aux noms de modes
grouped2_pivot = grouped2_pivot.rename(columns={
    0: "tramway",  # 0 -> tramway
    1: "metro",   # 1 -> metro
    2: "train",   # 2 -> train
    3: "bus",     # 3 -> bus
    6: "IGNORED", # 6 -> ignoré (autres modes)
    7: "IGNORED"  # 7 -> ignoré (autres modes)
})

# Aplatir les colonnes multi-index
grouped2_pivot.columns = [
    f"{name}_{mode}_1km"  # Format : passage_journalier_tramway_1km
    for name, mode in grouped2_pivot.columns
]

# Réinitialiser l'index pour avoir point_id comme colonne
grouped2_pivot = grouped2_pivot.reset_index()

# Supprimer les colonnes contenant '_nan_' ou 'IGNORED' (modes non pertinents)
grouped2_pivot = grouped2_pivot.loc[:, ~(grouped2_pivot.columns.str.contains('_nan_') | grouped2_pivot.columns.str.contains('IGNORED'))]

# Fusionner les résultats agrégés dans gdf_dvf pour créer gdf_dvf_final
gdf_dvf_final = gdf_dvf.merge(
    grouped2_pivot,
    on='point_id',  
    how='left'
)


## Ajout de la distance au centre de Paris

On ajoute la distance au centre de Paris, cela nous permet de nous rendre compte qu'une dizaine de points sont localisés en dehors de l'Île-de-France (sans doute des erreurs de l'API de géocodage). Nous supprimons ces points.

In [17]:
# Recuperatin des coordonnees du centre de paris (station Chatelet les Halles)
# Note : on utilise les df avec les geometries en metres

centre_paris = gdf_idfm_small_m[gdf_idfm_small_m['stop_name'] == 'Châtelet les Halles']["geometry"].values[0]

# ajouter une colonne distance au centre de paris

gdf_dvf_m['geometry'] = gdf_dvf_m.geometry.to_crs(epsg=3857)
gdf_dvf_m['dist_centre_paris_m'] = gdf_dvf_m.geometry.distance(centre_paris)

# ajout au dataframe final
gdf_dvf_final = gdf_dvf_final.merge(
    gdf_dvf_m[['point_id', 'dist_centre_paris_m']],
    on='point_id',
    how='left'
)

# filtrage des points trop éloignés (hors ile de france)

print(f"Suppression de {len(gdf_dvf_final[gdf_dvf_final['dist_centre_paris_m'] > 140000])} points trop éloignés (hors ile de france) :")

gdf_dvf_final = gdf_dvf_final[gdf_dvf_final['dist_centre_paris_m'] <= 140000]

Suppression de 10 points trop éloignés (hors ile de france) :


In [18]:
# nettoyage et enregistrement du GeoDataFrame final

gdf_dvf_final.drop(['search', 'Date mutation', 'Valeur fonciere', 'Code commune', 'result_score', 'point_id'], axis=1, inplace=True)

# ajout d'une colonne departement à gdf_dvf_final
gdf_dvf_final["Department"] = (gdf_dvf_final["Code_postal"].astype(int) // 1000).astype(str)

# reorganisation des colonnes
cols = gdf_dvf_final.columns.tolist()
cols_reordered = cols[:2] + cols[-2:]+ cols[7:8] + cols[2:7] + cols[8:-2]
gdf_dvf_final = gdf_dvf_final[cols_reordered]


# enregistrer le GeoDataFrame final en GeoJSON
gdf_dvf_final.to_file("cache/results/logements_transport_final.geojson", driver="GeoJSON", encoding="utf-8")

print(gdf_dvf_final[gdf_dvf_final['dist_centre_paris_m'] > 100000])

                             adresse Code_postal  dist_centre_paris_m  \
135                1 ALL DES MAZURES       77880        100053.071484   
373            1 AV DE LA LIBERATION       77130        104654.586876   
419             1 AV DES MARRONNIERS       91670        101040.791056   
589                      1 BD CARNOT       77570        125749.804118   
618                   1 CHE DE FLAGY       77940        115925.598151   
...                              ...         ...                  ...   
71921         93 RUE GRANDE VARENNES       77460        123845.828031   
72198           96 LES PETITES AUNES       77690        100197.091027   
72289  97 CHE DE L'ORME DE MONFERRAT       77320        106123.319618   
72405                98 RTE DE MORET       77140        101532.700910   
72542          99 RUE DESIRE THOISON       77130        110310.801259   

      Department  Valeur foncière au mètre carré                Commune  \
135           77                     3493.150685

## Ajout du revenu médian par commune 

In [26]:
path = "/home/onyxia/work/projet-python-ds/tests/idf_data2.xlsx"
df = pd.read_excel(path)
df = df[df['UNIT'] == "MEDIANE"]
df = df.rename(columns={"CODEGEO": "Code commune"})
gdf_dvf_final = gdf_dvf_final.merge(
    df[['Code commune', 'OBS_VALUE']],
    on='Code commune',
    how='left'
)

df

KeyError: 'Code commune'

## Visualisation du résultat

Pour illustre les données obtenues, on affiche le premier point de DVF. La carte interactive montre :
- Un marqueur bleu pour le point DVF.
- Des marqueurs rouges pour les arrêts de transport à moins de 1 km (avec passages journaliers agrégés par mode).
- Des marqueurs verts pour les arrêts les plus proches par mode de transport (bus, métro, tramway, train), avec distances en mètres.

In [12]:
from branca.element import Template, MacroElement
from script.leaflet_tools import FondCarteLeaflet
import folium
import great_tables as gt


# affichage des stops within radius on a map for the first point only
df = joined

m = FondCarteLeaflet(afficher_grande_couronne=True).get_map()

first_point = gdf_dvf.iloc[0]
folium.CircleMarker(
    location=[first_point.geometry.y, first_point.geometry.x],
    popup=first_point['point_id'],
    color='blue'
).add_to(m)
for _, row in df[df['point_id'] == first_point['point_id']].iterrows():
    stop = gdf_idfm_small[gdf_idfm_small['stop_id'] == row['stop_id']]

    stop0 = stop.iloc[0]
    route_name = stop0.get('stop_name', None)

    folium.Marker(
        location=[stop0.geometry.y, stop0.geometry.x],
        tooltip=f"Nearby stop:\n{route_name}",
        icon=folium.Icon(color='red', icon='info-sign')
    ).add_to(m)

m

for target in ["bus", "metro", "tramway", "train"]:
    stop_id = first_point.get(f"nearest_{target}_stop_id")
    stop = gdf_idfm_small[gdf_idfm_small['stop_id'] == stop_id]

    stop0 = stop.iloc[0]
    route_name = stop0.get('stop_name', None)

    folium.Marker(
        location=[stop0.geometry.y, stop0.geometry.x],
        tooltip=f"Nearest {target} stop:\n{route_name}",
        icon=folium.Icon(color='green', icon='info-sign')
    ).add_to(m)

# add legend once (do not recreate inside the loop)
legend_html = """
{% macro html(this, kwargs) %}
<div style="position: fixed; 
    bottom: 50px; left: 50px; width: 220px; padding:8px;
    border:2px solid grey; z-index:9999; font-size:14px;
    background-color:white; box-shadow:2px 2px 6px rgba(0,0,0,0.15);
    ">
  <b>Legend</b><br>
  <span style="display:inline-block;width:12px;height:12px;background:blue;border-radius:50%;margin-right:8px;vertical-align:middle;"></span>
    Point (DVF sample)<br>
  <span style="display:inline-block;width:12px;height:12px;background:red;border-radius:3px;margin-right:8px;vertical-align:middle;"></span>
    Nearby stop (within radius)<br>
  <span style="display:inline-block;width:12px;height:12px;background:green;border-radius:3px;margin-right:8px;vertical-align:middle;"></span>
    Nearest stop (by mode)<br>
  <hr style="margin:6px 0"/>
  <small>Hover or click markers for details</small>
</div>
{% endmacro %}
"""
legend = MacroElement()
legend._template = Template(legend_html)
m.get_root().add_child(legend)


# Sauvegarde de la carte 
from pathlib import Path

assets = Path("assets/maps")
assets.mkdir(parents=True, exist_ok=True)

m.save(assets / "main1.html")

[![Interactive map](/assets/maps/main1.png)](/assets/maps/main1.html)

In [13]:
from great_tables import GT
import numpy as np
import pandas as pd

# rebuild the transposed single-row table with groups


gdf_dvf_final_filtred = gdf_dvf_final.drop(columns=['nearest_tramway_stop_id', 'nearest_train_stop_id', 'nearest_bus_stop_id', 'nearest_metro_stop_id'])

groups = ["Localisation"] * 6 + ["Caractéristiques du logement"] * 5 + ["Distances des transports"] * 4 + ["Transports dans un rayon de 1km"] * (len(gdf_dvf_final_filtred.columns) - 15)

# compute stats for each column
rows = []
for col in gdf_dvf_final_filtred.columns:

    if pd.api.types.is_numeric_dtype(gdf_dvf_final_filtred[col]):
        min_s = f"{gdf_dvf_final_filtred[col].min():,.1f}"
        max_s = f"{gdf_dvf_final_filtred[col].max():,.1f}"
        mean_s = f"{gdf_dvf_final_filtred[col].mean():,.1f}"
        quartile_25 = f"{gdf_dvf_final_filtred[col].quantile(0.25):,.1f}"
        median = f"{gdf_dvf_final_filtred[col].median():,.1f}"
        quartile_75 = f"{gdf_dvf_final_filtred[col].quantile(0.75):,.1f}"
    else:
        try:
            min_s = str(gdf_dvf_final_filtred[col].min())
            max_s = str(gdf_dvf_final_filtred[col].max())
            # crop long strings for display
            if len(min_s) > 7:
                min_s = min_s[:4] + "..."
            if len(max_s) > 7:
                max_s = max_s[:4] + "..."
        except Exception:
            min_s = max_s = "-"
        mean_s = "-"
        cv_s = "-"
        quartile_25 = "-"
        median = "-"
        quartile_75 = "-"

    rows.append({"index": col, "Moyenne": mean_s, "Min": min_s, "25%": quartile_25, "50%": median, "75%": quartile_75, "Max": max_s,})

df_stats = pd.DataFrame(rows).assign(group=groups)

(
    GT(df_stats)
    .tab_stub(rowname_col="index", groupname_col="group")
    .tab_header("Statistiques descriptives des colonnes du jeu de données")
)

GT(_tbl_data=                             index   Moyenne      Min      25%       50%  \
0                          adresse         -  1 AL...        -         -   
1                      Code_postal         -    75001        -         -   
2              dist_centre_paris_m  26,702.7    137.3  8,717.4  19,904.9   
3                       Department         -       75        -         -   
4   Valeur foncière au mètre carré   5,539.7  1,000.0  3,104.9   4,488.9   
5                          Commune         -  ABLE...        -         -   
6              Surface reelle bati      72.8      9.0     40.0      65.0   
7                  Surface terrain     199.6      0.0      0.0       0.0   
8        Nombre pieces principales       3.3      1.0      2.0       3.0   
9                       Type local         -  Appa...        -         -   
10                        geometry         -        -        -         -   
11              nearest_bus_dist_m     251.9      0.3    145.5     221.0   
12            nearest_metro_dist_m  14,762.0      8.2    632.9   6,343.8   
13          nearest_tramway_dist_m   9,544.0     10.6  1,671.6   3,996.1   
14            nearest_train_dist_m   2,152.7     10.6    935.9   1,543.5   
15           nb_routes_tramway_1km       0.1      0.0      0.0       0.0   
16             nb_routes_metro_1km       0.7      0.0      0.0       0.0   
17             nb_routes_train_1km       0.4      0.0      0.0       0.0   
18               nb_routes_bus_1km       9.3      0.0      5.0       8.0   
19         nb_stations_tramway_1km       0.2      0.0      0.0       0.0   
20           nb_stations_metro_1km       1.1      0.0      0.0       0.0   
21           nb_stations_train_1km       0.3      0.0      0.0       0.0   
22             nb_stations_bus_1km       9.5      0.0      6.0       9.0   
23  passage_journalier_tramway_1km     103.1      0.0      0.0       0.0   
24    passage_journalier_metro_1km   1,069.8      0.0      0.0       0.0   
25    passage_journalier_train_1km     101.0      0.0      0.0       0.0   
26      passage_journalier_bus_1km   2,978.9      0.0    858.0   2,280.0   

         75%        Max                            group  
0          -    990 ...                     Localisation  
1          -      95880                     Localisation  
2   37,968.3  138,652.3                     Localisation  
3          -         95                     Localisation  
4    7,545.5   20,000.0                     Localisation  
5          -     YERRES                     Localisation  
6       91.0      300.0     Caractéristiques du logement  
7      262.0    9,940.0     Caractéristiques du logement  
8        4.0       20.0     Caractéristiques du logement  
9          -     Maison     Caractéristiques du logement  
10         -          -     Caractéristiques du logement  
11     313.7    4,261.1         Distances des transports  
12  22,284.3  119,312.3         Distances des transports  
13  11,771.2   92,561.1         Distances des transports  
14   2,453.3   35,427.2         Distances des transports  
15       0.0        3.0  Transports dans un rayon de 1km  
16       1.0        8.0  Transports dans un rayon de 1km  
17       1.0        9.0  Transports dans un rayon de 1km  
18      12.0       48.0  Transports dans un rayon de 1km  
19       0.0        6.0  Transports dans un rayon de 1km  
20       1.0       11.0  Transports dans un rayon de 1km  
21       1.0        3.0  Transports dans un rayon de 1km  
22      13.0       27.0  Transports dans un rayon de 1km  
23       0.0    3,329.0  Transports dans un rayon de 1km  
24     920.0   14,720.0  Transports dans un rayon de 1km  
25      87.0    2,506.0  Transports dans un rayon de 1km  
26   4,558.0   17,350.0  Transports dans un rayon de 1km  , _body=<great_tables._gt_data.Body object at 0x7f029e61e2c0>, _boxhead=Boxhead([ColInfo(var='index', type=<ColInfoTypeEnum.stub: 2>, column_label='index', column_align='left', column_width=None), ColInfo(va

In [14]:


# import geopandas as gpd

# import folium
# import branca.colormap as cm
# from numpy import log

# import os
# if os.getcwd().endswith("notebooks"):
#     os.chdir('..')
    
# from script.leaflet_tools import FondCarteLeaflet

# m = FondCarteLeaflet(afficher_grande_couronne=True).get_map()
# idf = gpd.read_file("cache/results/prix_logements.geojson")


# for _, row in gdf_dvf_final[gdf_dvf_final['dist_centre_paris_m'] > 140000].iterrows():
#     folium.CircleMarker(
#         location=[row.geometry.y, row.geometry.x], 
#         radius=0.5,
#         tooltip=f"{row['Valeur foncière au mètre carré']:.0f} €/m²",
#         color="blue",
#         fill=True,
#         fill_color="blue",
#         fill_opacity=0.7
#     ).add_to(m)
# # m
